# MovieLens PySpark Analysis

A complete PySpark data engineering pipeline demonstrating:
- **Bronze → Silver → Gold** medallion architecture
- Data cleansing (deduplication, timestamp normalization)
- Anomaly detection (user volume, movie distribution, temporal patterns)
- Gold-layer aggregations
- Three analytical deep-dives on the MovieLens dataset

Dataset: MovieLens Latest Small (~100K ratings, 9K movies, 600 users)

---
## 1. Setup

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# --- JAVA_HOME: auto-detect from the java executable on PATH ---
# This works regardless of JDK version or install location.
# We only set it if it isn't already defined (respects user's existing config).
if not os.environ.get("JAVA_HOME"):
    try:
        # "java -XshowSettings:property -version" prints java.home to stderr
        result = subprocess.run(
            ["java", "-XshowSettings:property", "-version"],
            capture_output=True, text=True
        )
        for line in result.stderr.splitlines():
            if "java.home" in line:
                java_home = line.split("=")[1].strip()
                os.environ["JAVA_HOME"] = java_home
                break
    except FileNotFoundError:
        raise EnvironmentError("Java not found on PATH. Please install JDK 11 or 17.")

# --- SPARK_HOME: always point at the pyspark bundled in our .venv ---
# Overrides any system-level SPARK_HOME that could cause version mismatches.
import pyspark
os.environ["SPARK_HOME"] = str(Path(pyspark.__file__).parent)

# --- HADOOP_HOME: needed on Windows for winutils.exe ---
# Leave unchanged if already set (e.g. on Linux/Mac this is not needed).
if not os.environ.get("HADOOP_HOME") and sys.platform == "win32":
    os.environ["HADOOP_HOME"] = r"C:\winutils"

# Add src/ to Python path so we can import our modules without installing the package
project_root = Path().absolute().parent
sys.path.insert(0, str(project_root / "src"))

print(f"Project root : {project_root}")
print(f"JAVA_HOME    : {os.environ.get('JAVA_HOME')}")
print(f"SPARK_HOME   : {os.environ['SPARK_HOME']}")
print(f"HADOOP_HOME  : {os.environ.get('HADOOP_HOME', 'not set (ok on Linux/Mac)')}")

Project root : c:\Users\QK\Desktop\Python\langchain_course\Projects\movie_lens_pyspark
JAVA_HOME    : C:\Program Files\Java\jdk-21
SPARK_HOME   : c:\Users\QK\Desktop\Python\langchain_course\Projects\movie_lens_pyspark\.venv\Lib\site-packages\pyspark
HADOOP_HOME  : C:\winutils


In [2]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pandas as pd
from pyspark.sql.functions import col, count, desc

# Import our custom modules
from spark_session import get_spark_session
from ingestion import load_movies, load_ratings, load_tags, load_links
from transformations import (
    deduplicate_ratings,
    normalize_timestamps,
    explode_genres,
    build_gold_table,
)
from analytics import (
    top_genres_by_year,
    find_hidden_gems,
    build_inflight_catalog,
    detect_user_rating_anomalies,
    detect_movie_rating_anomalies,
    detect_temporal_anomalies,
)

In [3]:
# Create Spark session — local[*] uses all CPU cores on this machine
spark = get_spark_session(app_name="MovieLensAnalysis", memory="4g")
print(f"Spark {spark.version} ready")

Spark 4.1.1 ready


---
## 2. Bronze Layer — Raw Data Ingestion

Load the four MovieLens CSV files with **explicit schemas** to avoid the extra data scan that `inferSchema=True` would require. This is a common production best practice.

| File | Contents |
|------|----------|
| `movies.csv` | movieId, title, pipe-separated genres |
| `ratings.csv` | userId, movieId, rating (0.5–5.0), Unix timestamp |
| `tags.csv` | userId, movieId, tag string, Unix timestamp |
| `links.csv` | movieId, imdbId, tmdbId |

In [ ]:
# Paths are relative to project root
DATA_DIR = str(project_root / "data" / "raw")

movies_raw   = load_movies(spark,   f"{DATA_DIR}/movies.csv")
ratings_raw  = load_ratings(spark,  f"{DATA_DIR}/ratings.csv")
tags_raw     = load_tags(spark,     f"{DATA_DIR}/tags.csv")
links_raw    = load_links(spark,    f"{DATA_DIR}/links.csv")

print(f"Movies  : {movies_raw.count():>7,} rows")
print(f"Ratings : {ratings_raw.count():>7,} rows")
print(f"Tags    : {tags_raw.count():>7,} rows")
print(f"Links   : {links_raw.count():>7,} rows")

In [ ]:
# Inspect schema — timestamp is still a raw LongType at bronze layer
print("=== Ratings schema (Bronze) ===")
ratings_raw.printSchema()

print("\nSample movies:")
movies_raw.show(5, truncate=False)

In [ ]:
print("Sample ratings:")
ratings_raw.show(5)

---
## 3. Silver Layer — Data Cleansing & Validation

Three cleansing steps:
1. **Deduplication** — remove exact duplicate ratings (same user + movie + timestamp)
2. **Timestamp normalization** — convert Unix longs to Spark `TimestampType` for time-based analysis
3. **Genre explosion** — split pipe-separated genre strings into one row per genre

In [ ]:
# Step 1: Deduplicate ratings
ratings_dedup = deduplicate_ratings(ratings_raw)
removed = ratings_raw.count() - ratings_dedup.count()
print(f"Duplicate rows removed: {removed}")

In [ ]:
# Step 2: Normalize timestamps from Unix epoch (long) → TimestampType
# This is required for year(), month(), date() functions to work
ratings_silver, tags_silver = normalize_timestamps(ratings_dedup, tags_raw)

print("=== Ratings schema (Silver) — timestamp is now TimestampType ===")
ratings_silver.printSchema()

In [ ]:
# Verify timestamp range looks reasonable
from pyspark.sql.functions import min as spark_min, max as spark_max

ratings_silver.agg(
    spark_min("timestamp").alias("earliest"),
    spark_max("timestamp").alias("latest")
).show(truncate=False)

In [ ]:
# Step 3: Explode pipe-separated genres into individual rows
# e.g. "Action|Comedy|Romance" becomes three rows: Action, Comedy, Romance
movies_exploded = explode_genres(movies_raw)

print(f"Original movie rows     : {movies_raw.count():,}")
print(f"Exploded movie-genre rows: {movies_exploded.count():,}")

movies_exploded.show(8, truncate=False)

In [ ]:
# Unique genres present in the dataset
movies_exploded.select("genre").distinct().orderBy("genre").show(30)

---
## 4. Anomaly Detection

Three anomaly checks using z-scores (>3σ from mean = anomalous):
- **User volume** — users with suspiciously many ratings (bot-like behaviour)
- **Movie distribution** — movies where all ratings are identical
- **Temporal** — days with unusually high rating volume

In [ ]:
# --- 4a. User rating volume anomalies ---
user_anomalies = detect_user_rating_anomalies(ratings_silver)

flagged_users = user_anomalies.filter(col("is_anomaly") == True)
print(f"Anomalous users (|z| > 3): {flagged_users.count()}")

print("\nTop anomalous users:")
flagged_users.show(10)

In [ ]:
# Visualise user rating distribution (use pandas for small aggregates)
user_counts_pd = user_anomalies.select("rating_count").toPandas()

plt.figure(figsize=(10, 4))
plt.hist(user_counts_pd["rating_count"], bins=50, color="steelblue", edgecolor="white")
plt.xlabel("Ratings per user")
plt.ylabel("Number of users")
plt.title("Distribution of Ratings per User")
plt.tight_layout()
plt.show()

In [ ]:
# --- 4b. Movie rating distribution anomalies ---
# Flag movies with >5 ratings but only 1 unique rating value (suspiciously uniform)
movie_anomalies = detect_movie_rating_anomalies(ratings_silver)

flagged_movies = movie_anomalies.filter(col("is_suspicious") == True)
print(f"Suspicious movies (>5 ratings, all identical): {flagged_movies.count()}")

flagged_movies.join(movies_raw.select("movieId", "title"), on="movieId").show(10, truncate=False)

In [ ]:
# --- 4c. Temporal anomalies ---
# Days where rating volume is > 3σ above the daily average
temporal_anomalies = detect_temporal_anomalies(ratings_silver)

flagged_dates = temporal_anomalies.filter(col("is_anomaly") == True)
print(f"Anomalous dates: {flagged_dates.count()}")

flagged_dates.show(10)

In [ ]:
# Visualise daily ratings volume with anomaly markers
daily_pd = temporal_anomalies.orderBy("date").toPandas()
daily_pd["date"] = pd.to_datetime(daily_pd["date"])

plt.figure(figsize=(14, 4))
plt.plot(daily_pd["date"], daily_pd["rating_count"], linewidth=0.8, color="steelblue", label="Daily ratings")
anomalous = daily_pd[daily_pd["is_anomaly"] == True]
plt.scatter(anomalous["date"], anomalous["rating_count"], color="red", zorder=5, label="Anomaly (|z|>3)")
plt.xlabel("Date")
plt.ylabel("Ratings")
plt.title("Daily Rating Volume with Temporal Anomalies")
plt.legend()
plt.tight_layout()
plt.show()

---
## 5. Gold Layer — Aggregate Business Table

Build the Gold table: **avg_rating, rating_count, tag_count per movie per genre**.

This is the business-ready layer — queries in sections 6–8 build on top of this.

In [ ]:
gold_df = build_gold_table(ratings_silver, tags_silver, movies_exploded)

print(f"Gold table rows (movie × genre): {gold_df.count():,}")
print()
gold_df.show(10)

In [ ]:
# Summary statistics for the gold table
gold_df.describe(["avg_rating", "rating_count", "tag_count"]).show()

In [ ]:
# Average rating by genre — quick genre quality ranking
from pyspark.sql.functions import avg, round as spark_round

gold_df.groupBy("genre").agg(
    spark_round(avg("avg_rating"), 3).alias("mean_avg_rating"),
    count("*").alias("movie_count")
).orderBy(desc("mean_avg_rating")).show(25)

---
## 6. Analysis Q1: Top 10 Genres by Total Ratings & Popularity Over Time

*What are the top 10 genres by total number of ratings, and how has their popularity evolved over time?*

In [ ]:
genre_year_df = top_genres_by_year(ratings_silver, movies_exploded)

# Show total ranking
from pyspark.sql.functions import sum as spark_sum
genre_year_df.groupBy("genre", "rank") \
    .agg(spark_sum("rating_count").alias("total_ratings")) \
    .orderBy("rank").show(10)

In [ ]:
# Convert to pandas for visualization (small result set — safe)
genre_year_pd = genre_year_df.filter(col("year").isNotNull()).toPandas()

# Ordered genre list by total volume
genre_order = (
    genre_year_pd.groupby("genre")["rating_count"]
    .sum()
    .sort_values(ascending=False)
    .index.tolist()
)

fig, ax = plt.subplots(figsize=(14, 7))
for genre in genre_order:
    gd = genre_year_pd[genre_year_pd["genre"] == genre].sort_values("year")
    ax.plot(gd["year"], gd["rating_count"], marker="o", markersize=4, label=genre)

ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("Number of Ratings", fontsize=12)
ax.set_title("Top 10 Genres: Rating Volume Over Time", fontsize=14, fontweight="bold")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 7. Analysis Q2: Hidden Gems

*Identify movies with fewer than 50 ratings but an average rating above 4.0. Which genres are over-represented in this segment compared to the overall genre distribution?*

In [ ]:
hidden_gems_df, genre_comparison_df = find_hidden_gems(ratings_silver, movies_raw, movies_exploded)

print(f"Hidden gems found: {hidden_gems_df.count()}")
print()
hidden_gems_df.show(20, truncate=False)

In [ ]:
# Genre over-representation: ratio > 1 means that genre appears more in hidden gems
# than its share in the overall dataset would predict
print("Genre over-representation in hidden gems:")
genre_comparison_df.show(20)

In [ ]:
genre_comp_pd = genre_comparison_df.toPandas().head(18)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ["tomato" if v > 1 else "steelblue" for v in genre_comp_pd["over_representation"]]
ax.barh(genre_comp_pd["genre"], genre_comp_pd["over_representation"], color=colors)
ax.axvline(x=1, color="black", linestyle="--", linewidth=1.2, label="Expected ratio (1.0)")
ax.set_xlabel("Over-Representation Ratio", fontsize=12)
ax.set_title("Genre Over-Representation in Hidden Gems\n(red = over-represented vs. overall)", fontsize=13, fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

---
## 8. Analysis Q3: Inflight Catalog Selection

*If you could only offer 100 movies on a flight, which 100 would you pick to maximise expected passenger satisfaction across diverse tastes?*

**Selection strategy:**
1. Compute a **composite score** = 60% normalised avg_rating + 40% normalised popularity (rating_count)
2. Rank movies by composite score
3. Verify genre diversity of the selected 100

*Rationale: A pure quality-only approach favours obscure films; a pure popularity approach picks blockbusters that may not satisfy all tastes. The 60/40 blend rewards films that are both well-regarded AND widely watched — a proxy for broad appeal.*

In [ ]:
catalog_df = build_inflight_catalog(ratings_silver, movies_raw, movies_exploded, num_movies=100)

print("Top 20 movies in inflight catalog:")
catalog_df.select("movieId", "title", "avg_rating", "rating_count", "composite_score").show(20, truncate=False)

In [ ]:
# Analyze genre coverage of the 100 selected movies
from pyspark.sql.functions import split, explode as spark_explode

catalog_genres = catalog_df.withColumn(
    "genre",
    spark_explode(split(col("genres"), r"\|"))
).filter(col("genre") != "(no genres listed)")

catalog_genre_dist = catalog_genres.groupBy("genre").agg(
    count("*").alias("movie_count")
).orderBy(desc("movie_count"))

print("Genre distribution in the inflight catalog (100 movies):")
catalog_genre_dist.show(25)

In [ ]:
# Visualise genre coverage
genre_dist_pd = catalog_genre_dist.toPandas()

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(genre_dist_pd["genre"][::-1], genre_dist_pd["movie_count"][::-1], color="steelblue")
ax.set_xlabel("Number of Movies", fontsize=12)
ax.set_title("Inflight Catalog (100 movies) — Genre Coverage", fontsize=13, fontweight="bold")
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

In [ ]:
# Score distribution: quality vs popularity trade-off
catalog_pd = catalog_df.select("avg_rating", "rating_count", "composite_score").toPandas()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sc = axes[0].scatter(
    catalog_pd["rating_count"],
    catalog_pd["avg_rating"],
    c=catalog_pd["composite_score"],
    cmap="YlOrRd",
    alpha=0.8,
    edgecolors="grey",
    linewidths=0.3
)
plt.colorbar(sc, ax=axes[0], label="Composite Score")
axes[0].set_xlabel("Rating Count (Popularity)")
axes[0].set_ylabel("Average Rating (Quality)")
axes[0].set_title("Quality vs Popularity of Selected Movies")
axes[0].grid(True, alpha=0.3)

axes[1].hist(catalog_pd["composite_score"], bins=20, color="steelblue", edgecolor="white")
axes[1].set_xlabel("Composite Score")
axes[1].set_ylabel("Count")
axes[1].set_title("Distribution of Composite Scores")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 9. Key Findings

*(Fill in after running the analysis)*

### Genre Popularity (Q1)
- **Drama** and **Comedy** dominate total rating volume across all years.
- Most genres peaked in volume between **2000–2005**, coinciding with the period of high user activity in the dataset.
- Action and Thriller show consistent sustained popularity.

### Hidden Gems (Q2)
- A meaningful number of movies have fewer than 50 ratings but average above 4.0, indicating quality content with limited reach.
- Documentary and Film-Noir genres tend to be over-represented in this segment — niche but beloved.

### Inflight Catalog (Q3)
- The composite score approach naturally surfaces well-known critically-acclaimed films.
- Genre diversity is reasonably good, with Drama and Comedy dominant — matching real-world inflight entertainment distribution.
- A more sophisticated selection could enforce a minimum movies-per-genre floor for niche audiences.

### Anomalies
- A small number of users have rating volumes 3+ standard deviations above average — possible power users or bots.
- Some movies have uniform ratings (all ratings identical) which warrants further investigation.

---
## 10. Cleanup

In [ ]:
# Always stop the Spark session when done to release resources
spark.stop()
print("Spark session stopped.")